# Pool de escalabilidad (E5-base)

Vectoriza hasta 1 millón de abstracts en tramos de 50 000, de unos 150 MB cada uno. Antes de empezar
mide el espacio libre en Drive y **se detiene antes de llenarlo**.

**Cómo usarlo**

1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**.
2. Menú **Entorno de ejecución → Ejecutar todo**, y autoriza el acceso a Google Drive cuando lo pida.
3. Deja la pestaña abierta, con la tapa de la laptop abierta y sin suspender el equipo.

**Si se corta o se acaba la GPU gratis:** vuelve a **Ejecutar todo**, aunque sea otro día. Lo que ya está
hecho se salta. Con 500 000 abstracts (10 tramos) ya es suficiente; puedes parar ahí.

Cada línea de avance indica cuánto tardó ese tramo. Todo queda en `tabd_scale_e5_base/` en Google Drive.

In [ ]:
#@title 1. Preparar el entorno
import base64, glob, io, os, shutil, zipfile
import torch
from google.colab import drive

assert torch.cuda.is_available(), "No hay GPU: elige GPU T4 en Entorno de ejecución y vuelve a ejecutar."
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/tabd_scale_e5_base"
os.makedirs(OUT, exist_ok=True)

BUNDLE = """
...
"""
os.makedirs("/content/tabd", exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall("/content/tabd")
os.chdir("/content/tabd")
print("Archivos listos en", os.getcwd())

In [ ]:
#@title 2. Instalar dependencias
!pip install -q datasets sentence-transformers pandas pyarrow

In [ ]:
#@title 3. Ver cuántos tramos caben en Drive
CHUNK_VECTORS = 50_000
CHUNK_BYTES = CHUNK_VECTORS * 768 * 4
MARGIN_BYTES = 300 * 2**20

done = len(glob.glob(f"{OUT}/part_e5_base_*.npy"))
free = shutil.disk_usage("/content/drive/MyDrive").free
fits = max(0, free - MARGIN_BYTES) // CHUNK_BYTES
parts = min(20, done + fits)
print(f"Libre en Drive: {free / 2**30:.2f} GB | tramos hechos: {done} | caben {fits} más")
assert parts > 0, "No hay espacio libre en Drive: libera al menos 500 MB y vuelve a ejecutar."
STOP_AFTER = parts * CHUNK_VECTORS
print(f"Objetivo: {STOP_AFTER:,} abstracts ({parts} tramos)")

In [ ]:
#@title 4. Vectorizar (varias horas; se puede retomar)
!python colab_embed_scale.py --model e5_base --n 1000000 --stop-after {STOP_AFTER} --batch-size 64 --out "{OUT}"

In [ ]:
#@title 5. Terminar de subir a Drive
drive.flush_and_unmount()
print("Drive sincronizado. Revisa la carpeta tabd_scale_e5_base y avísame cuántos tramos quedaron.")